In [20]:
#spark.conf.set(
#    "fs.azure.sas.fabmigration.fabmigration.dfs.core.windows.net", 
#    "sp=racwdlmeop&st=2026-01-30T16:00:00Z&se=2027-01-31T04:00:00Z&spr=https&sv=2024-11-04&sr=c&sig=EJ%2BzeyUvoM48uvMfwIledAfb7caaPb4LCfQY%2BDigsFw%3D"
#    )

#spark.conf.set("fs.azure.account.auth.type.fabmigration.dfs.core.windows.net", "SAS")
#spark.conf.set("fs.azure.sas.token.provider.type.fabmigration.dfs.core.windows.net", "com.microsoft.azure.synapse.tokenlibrary.ConfBasedSASProvider")
#spark.conf.set("spark.storage.synapse.fabmigration.fabmigration.dfs.core.windows.net.sas", "sp=racwdlmeop&st=2026-01-30T16:00:00Z&se=2027-01-31T04:00:00Z&spr=https&sv=2024-11-04&sr=c&sig=EJ%2BzeyUvoM48uvMfwIledAfb7caaPb4LCfQY%2BDigsFw%3D")

#df = spark.read.parquet(Sourcepath)

#display(df.limit(10))

StatementMeta(fabmigspark, 16, 21, Finished, Available, Finished)

In [133]:
ADLSaccount = "fabmigration"
containerName = "fabmigration"
sourceFolder = "bronze/adventureworks2"
targetFolder = "silver/adventureworks"
targetEnv = "silver"
entityName = "Sales.Store"
fileExtension = "parquet"

StatementMeta(fabmigspark, 72, 157, Finished, Available, Finished)

In [134]:
sourcePath = f'abfss://{containerName}@{ADLSaccount}.dfs.core.windows.net/{sourceFolder}/{entityName}.{fileExtension}'
targetPath = f'abfss://{containerName}@{ADLSaccount}.dfs.core.windows.net/{targetFolder}/'

print(sourcePath)
print(targetPath)

StatementMeta(fabmigspark, 72, 158, Finished, Available, Finished)

abfss://fabmigration@fabmigration.dfs.core.windows.net/bronze/adventureworks2/Sales.Store.parquet
abfss://fabmigration@fabmigration.dfs.core.windows.net/silver/adventureworks/


In [135]:
df = spark.read.load(sourcePath, format='parquet')
#display(df.limit(10))
df.show()

StatementMeta(fabmigspark, 72, 159, Finished, Available, Finished)

+----------------+--------------------+-------------+--------------------+--------------------+--------------------+
|BusinessEntityID|                Name|SalesPersonID|        Demographics|             rowguid|        ModifiedDate|
+----------------+--------------------+-------------+--------------------+--------------------+--------------------+
|             292|Next-Door Bike Store|          279|<StoreSurvey xmln...|a22517e3-848d-4eb...|2014-09-12 11:15:...|
|             294|Professional Sale...|          276|<StoreSurvey xmln...|b50ca50b-c601-4a1...|2014-09-12 11:15:...|
|             296|      Riders Company|          277|<StoreSurvey xmln...|337c3688-1339-4e1...|2014-09-12 11:15:...|
|             298|  The Bike Mechanics|          275|<StoreSurvey xmln...|7894f278-f0c8-4d1...|2014-09-12 11:15:...|
|             300|   Nationwide Supply|          286|<StoreSurvey xmln...|c3fc9705-a8c4-4f3...|2014-09-12 11:15:...|
|             302|Area Bike Accesso...|          281|<StoreSurve

In [136]:
%run /RetailSilver/00_Functions

StatementMeta(fabmigspark, 72, 161, Finished, Available, Finished)

In [137]:
# Read parquet file to get data
dfdata = spark.read.load(sourcePath, format='parquet')

# Set the schema for entity. Also if needed the first row.
entitySchema = GetSaleTableSchema(entityName)
firstRow = GetSaleTableFirstRow(entityName, dfdata)

# Create empty DF with schema and first row (if needed)
df = spark.createDataFrame(firstRow, entitySchema)

# Update column names from parquet file if needed
dfdata = UpdateDFColNames(entityName, dfdata)

#Join empty Df with data from parquet
df = df.unionByName(dfdata)

display(df.limit(10))

StatementMeta(fabmigspark, 72, 162, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6b53fe95-a47d-4eea-bf43-642ddf03747c)

In [138]:
# Clean
df = clean_df(df)

StatementMeta(fabmigspark, 72, 163, Finished, Available, Finished)

In [139]:
# Set name for silver
entityNameFix = f'silver_{entityName.replace(".", "_")}'
#print(entityNameFix)

# Save DF into stage after clean process
SaveDF(df, targetEnv, entityNameFix, targetPath)

StatementMeta(fabmigspark, 72, 164, Finished, Available, Finished)

In [26]:
#df = spark.read.load(sourcePath, format='parquet')
#df = spark.read.parquet(sourcePath)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType BooleanType, DateType, TimestampType

schemaCountryRegionCurrency = StructType([
    StructField("CountryRegionCode", StringType(), True), 
    StructField("Name", StringType(), True), 
    StructField("ModifiedDate", StringType(), True)
    ])
#.schema(schemaCountryRegionCurrency)

dfdata = spark.read.load(sourcePath, format='parquet')

firstRow = [(dfdata.columns[0], dfdata.columns[1], dfdata.columns[2])]
df = spark.createDataFrame(firstRow, schemaCountryRegionCurrency)

dfdata = dfdata.toDF("CountryRegionCode", "Name", "ModifiedDate")

df = df.unionByName(dfdata)

display(df.limit(10))
#df.show()

StatementMeta(fabmigspark, 19, 27, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, acc0034d-fa22-4e89-b849-0bf964db1791)

In [27]:
# Change entity name schema into folder then form full path to save the delta table
targetFullPath = f'{targetPath}{entityName.replace(".", "/")}'

print(targetFullPath)

#Save data frame into Lake
#df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(targetFullPath)

StatementMeta(fabmigspark, 55, 20, Finished, Available, Finished)

abfss://fabmigration@fabmigration.dfs.core.windows.net/bronze/adventureworksStage/Sales/CountryRegionCurrency


In [31]:
# Create schema if not exist
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {targetEnv}""")

#Register table to use in Synapse
spark.sql(f""" CREATE TABLE IF NOT EXISTS {targetEnv}.{entityName.replace(".", "_").lower()} USING DELTA LOCATION '{targetFullPath}/' """)

StatementMeta(fabmigspark, 55, 24, Finished, Available, Finished)

DataFrame[]

In [23]:
%%sql
DROP TABLE silver.sales_customer

StatementMeta(fabmigspark, 72, 30, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [15]:
%%sql
DROP TABLE stage.sales_currencyrate;

DROP TABLE stage.sales_currency;

DROP TABLE stage.sales_countryregioncurrency;

StatementMeta(fabmigspark, 72, 20, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%sql

SELECT
      crr.CurrencyCode
    , crr.Name
    , ifnull(crc.CountryRegionCode, 'XX') AS CountryRegionCode
FROM stage.sales_currency crr
LEFT JOIN stage.sales_countryregioncurrency crc ON
    crc.CurrencyCode = crr.CurrencyCode

StatementMeta(fabmigspark, 58, 8, Finished, Available, Finished)

<Spark SQL result set with 117 rows and 3 fields>

In [9]:
query = """
SELECT
      crr.CurrencyCode
    , crr.Name
    , ifnull(crc.CountryRegionCode, 'XX') AS CountryRegionCode
FROM stage.sales_currency crr
LEFT JOIN stage.sales_countryregioncurrency crc ON
    crc.CurrencyCode = crr.CurrencyCode
"""

#print(query)

newdf = spark.sql(query)

display(newdf)

StatementMeta(fabmigspark, 58, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 2215401c-2ebf-4a48-aa42-86b9049b4380)

In [13]:
%%sql
SELECT * FROM silver.currency

StatementMeta(, , -1, Cancelled, , Cancelled)

In [ ]:
ADLSaccount = "fabmigration"
containerName = "fabmigration"
targetFolder = "Config"
entityName = "tbl_Entities"